# Virtual Robot Race - AI Training on Google Colab

This notebook trains the Virtual Robot Race AI model on Google Colab.

## Prerequisites

1. **Upload to Google Drive (manual)**
   - `Robot1/training_data/` folder → `/MyDrive/virtual-robot-race/training_data/`
   - `Robot1/model.py` file → `/MyDrive/virtual-robot-race/model.py` ⚠️ Important

2. Upload this notebook to Google Drive
   - Upload destination: `/MyDrive/virtual-robot-race/train_on_colab.ipynb`

3. Open with Google Colab
   - Double-click file in Google Drive → "Open with Google Colaboratory"

4. Runtime → Change runtime type → Select **GPU**

## Workflow

1. **Cell 1-2**: Environment setup (mount Google Drive, install libraries)
2. **Cell 3**: Create iteration folder
3. **Cell 4**: Select and copy training data (automatically uses all run_ folders)
4. **Cell 5**: Generate dataset manifest
5. **Cell 6-7**: Train model
6. **Cell 8**: Visualize results
7. **Manual**: Download `model.pth` to local `Robot1/models/`

---
## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Google Drive をマウント
drive.mount('/content/drive')

# プロジェクトルート設定
PROJECT_ROOT = "/content/drive/MyDrive/virtual-robot-race"
TRAINING_DATA_ROOT = os.path.join(PROJECT_ROOT, "training_data")
EXPERIMENTS_ROOT = os.path.join(PROJECT_ROOT, "experiments")
ITERATIONS_ROOT = os.path.join(EXPERIMENTS_ROOT, "iterations")

# フォルダ存在確認
if not os.path.exists(TRAINING_DATA_ROOT):
    print(f"⚠️ WARNING: {TRAINING_DATA_ROOT} が見つかりません")
    print(f"   ローカルの Robot1/training_data/ を Google Drive にアップロードしてください")
else:
    run_folders = [f for f in os.listdir(TRAINING_DATA_ROOT) if f.startswith('run_')]
    print(f"✓ Training data found: {len(run_folders)} run folders")

# experiments/iterations フォルダ作成
os.makedirs(ITERATIONS_ROOT, exist_ok=True)
print(f"\n✓ Project root: {PROJECT_ROOT}")
print(f"✓ Experiments root: {EXPERIMENTS_ROOT}")

---
## Cell 2: Install libraries

In [ ]:
import sys
import torch
import torchvision

# 既にインストール済みのライブラリを確認
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

# CUDA確認
if torch.cuda.is_available():
    print(f"\n✓ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ WARNING: CUDA not available. ランタイム → ランタイムのタイプを変更 → GPU を選択してください")

# 追加で必要なライブラリがあればここでインストール
# !pip install [package_name]

---
## Cell 3: Create iteration folder

In [ ]:
from datetime import datetime
from pathlib import Path

# Iteration名生成（ローカルと同じ形式）
timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
iteration_name = f"iteration_{timestamp}"
iteration_path = Path(ITERATIONS_ROOT) / iteration_name

# フォルダ構造作成
iteration_path.mkdir(parents=True, exist_ok=True)
data_sources_path = iteration_path / "data_sources"
data_sources_path.mkdir(exist_ok=True)
plots_path = iteration_path / "plots"
plots_path.mkdir(exist_ok=True)

print(f"✓ Created iteration: {iteration_name}")
print(f"  Path: {iteration_path}")
print(f"\n📋 Next step: 次のセルで学習データを選択してください")

---
## Cell 4: Automatically copy all training data

Automatically copies all available `run_` folders to `data_sources/`.

In [ ]:
import shutil
import json

# 利用可能なrun_フォルダをリストアップ
available_runs = sorted([f for f in os.listdir(TRAINING_DATA_ROOT) if f.startswith('run_')])

print(f"利用可能な run_ フォルダ ({len(available_runs)} 個):")
print("-" * 80)
for i, run_folder in enumerate(available_runs, 1):
    run_path = Path(TRAINING_DATA_ROOT) / run_folder
    
    # metadata.csv のサンプル数を確認
    csv_file = run_path / "metadata.csv"
    if csv_file.exists():
        with open(csv_file, 'r') as f:
            num_lines = sum(1 for _ in f) - 1  # ヘッダー除く
        print(f"{i:2d}. {run_folder} ({num_lines} samples)")
    else:
        print(f"{i:2d}. {run_folder} (metadata.csv not found)")

print("\n" + "=" * 80)
print(f"📂 全ての run_ フォルダ ({len(available_runs)} 個) を data_sources/ にコピーします")
print("=" * 80)

# 全てのrun_を自動的にコピー
selected_runs = available_runs

# data_sources/ にコピー
print(f"\n📂 {len(selected_runs)} 個の run_ をコピー中...")
for run_folder in selected_runs:
    src = Path(TRAINING_DATA_ROOT) / run_folder
    dst = data_sources_path / run_folder
    
    if dst.exists():
        print(f"  ⏭️  {run_folder} (already exists, skipping)")
    else:
        shutil.copytree(src, dst)
        print(f"  ✓ {run_folder}")

print(f"\n✓ データコピー完了: {data_sources_path}")
print(f"  合計 {len(selected_runs)} 個の run_ フォルダ")

---
## Cell 5: Generate dataset manifest

Analyzes data in `data_sources/` and generates `dataset_manifest.json`.

In [ ]:
import csv
import json
from pathlib import Path
from datetime import datetime

def create_dataset_manifest(data_sources_path, output_path):
    """
    data_sources/ 内の run_ フォルダを分析し、dataset_manifest.json を生成
    """
    manifest = {
        "created_at": datetime.now().isoformat(),
        "runs": [],
        "total_samples": 0,
        "completed_runs": 0,
        "incomplete_runs": 0
    }
    
    run_folders = sorted([f for f in data_sources_path.iterdir() if f.name.startswith('run_')])
    
    print(f"Analyzing {len(run_folders)} run folders...\n")
    
    for run_folder in run_folders:
        csv_file = run_folder / "metadata.csv"
        
        if not csv_file.exists():
            print(f"⚠️  {run_folder.name}: metadata.csv not found, skipping")
            continue
        
        # CSVデータ読み込み
        with open(csv_file, 'r') as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        
        if len(rows) == 0:
            print(f"⚠️  {run_folder.name}: Empty CSV, skipping")
            continue
        
        # 統計情報取得（metadata.csvのカラム名に対応）
        # status が "Lap0", "Lap1", "Lap2", ... のフレームを学習対象とする
        # StartSequence, Finish, Fallen, FalseStart は除外
        racing_rows = [
            row for row in rows 
            if row.get('status', '').startswith('Lap')
        ]
        
        if len(racing_rows) == 0:
            print(f"⚠️  {run_folder.name}: No Lap frames found, skipping")
            continue
        
        # レース完走判定（Lapカウントで判定）
        # Unity の仕様: Lap0=1周目走行中, Lap1=1周完了(2周目走行中), Lap2=2周完了(3周目走行中)
        # 2周完走 = Lap1 が存在する
        all_statuses = set(row.get('status') for row in racing_rows)
        max_lap = max([int(s.replace('Lap', '')) for s in all_statuses if s.startswith('Lap')], default=0)
        race_completed = (max_lap >= 1)  # Lap1以上なら2周完走
        
        # レースタイム取得
        race_time_ms = float(racing_rows[-1].get('race_time_ms', 0))
        race_time_sec = race_time_ms / 1000.0
        
        # drive_torque/steer_angle の統計
        torques = [float(row['drive_torque']) for row in racing_rows]
        steers = [float(row['steer_angle']) for row in racing_rows]
        
        avg_torque = sum(torques) / len(torques)
        avg_steer = sum(steers) / len(steers)
        
        run_info = {
            "run_id": run_folder.name,
            "num_samples": len(racing_rows),
            "race_completed": race_completed,
            "max_lap": max_lap,
            "race_time_sec": race_time_sec,
            "avg_torque": round(avg_torque, 4),
            "avg_steer": round(avg_steer, 4)
        }
        
        manifest["runs"].append(run_info)
        manifest["total_samples"] += len(racing_rows)
        
        if race_completed:
            manifest["completed_runs"] += 1
            status = f"✓ COMPLETED (Lap{max_lap})"
        else:
            manifest["incomplete_runs"] += 1
            status = f"⚠️ INCOMPLETE (Lap{max_lap})"
        
        print(f"{status} {run_folder.name}: {len(racing_rows)} samples, {race_time_sec:.1f}s")
    
    # manifest保存
    with open(output_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    return manifest

# Manifest生成
manifest_path = iteration_path / "dataset_manifest.json"
manifest = create_dataset_manifest(data_sources_path, manifest_path)

print("\n" + "=" * 80)
print("📊 Dataset Summary")
print("=" * 80)
print(f"Total runs: {len(manifest['runs'])}")
print(f"  Completed: {manifest['completed_runs']}")
print(f"  Incomplete: {manifest['incomplete_runs']}")
print(f"Total samples: {manifest['total_samples']:,}")
print(f"\n✓ Manifest saved: {manifest_path}")

---
## Cell 6: Model definition and dataset preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import csv
from pathlib import Path
import sys
import random
import numpy as np

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# ===========================
# Seed設定（再現性のため）
# ===========================
def set_seed(seed=42):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Random seed: {SEED}")

# ===========================
# model.py をインポート
# ===========================
MODEL_FILE = Path(PROJECT_ROOT) / "model.py"

if not MODEL_FILE.exists():
    print(f"⚠️ ERROR: {MODEL_FILE} が見つかりません")
    print(f"   ローカルの Robot1/model.py を Google Drive にアップロードしてください")
    print(f"   アップロード先: /MyDrive/virtual-robot-race/model.py")
    raise FileNotFoundError(f"model.py not found at {MODEL_FILE}")

# Python のインポートパスに追加
sys.path.insert(0, str(MODEL_FILE.parent))

# model.py から DrivingNetwork をインポート
from model import DrivingNetwork

print(f"✓ model.py imported from: {MODEL_FILE}")

# ===========================
# Dataset定義（train.pyと同じフィルタリング）
# ===========================
class RobotRaceDataset(Dataset):
    # train.pyと同じ: Lap0, Lap1, Lap2, Finish を使用
    VALID_RACING_STATUS = ["Lap0", "Lap1", "Lap2", "Finish"]
    
    def __init__(self, data_sources_path, transform=None, exclude_start_sequence=True):
        self.data_sources_path = Path(data_sources_path)
        self.transform = transform
        self.samples = []
        
        # 全run_フォルダからサンプル収集
        run_folders = sorted([f for f in self.data_sources_path.iterdir() if f.name.startswith('run_')])
        
        print(f"\nLoading dataset from {len(run_folders)} run folders...")
        print("-" * 80)
        
        total_loaded = 0
        total_skipped = 0
        
        for run_folder in run_folders:
            csv_file = run_folder / "metadata.csv"
            if not csv_file.exists():
                print(f"⚠️  {run_folder.name}: metadata.csv not found, skipping")
                continue
            
            loaded_count = 0
            skipped_count = 0
            skipped_reasons = {}
            
            with open(csv_file, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    status = row.get('status', '')
                    
                    # Filter: Use racing data (Lap0, Lap1, Lap2, Finish)
                    if exclude_start_sequence and status not in self.VALID_RACING_STATUS:
                        skipped_count += 1
                        skipped_reasons[status] = skipped_reasons.get(status, 0) + 1
                        continue
                    
                    # filename カラムから画像パスを取得
                    filename = row['filename']
                    image_path = run_folder / "images" / filename
                    
                    if image_path.exists():
                        self.samples.append({
                            'image_path': str(image_path),
                            'drive_torque': float(row['drive_torque']),
                            'steer_angle': float(row['steer_angle']),
                            'soc': float(row['soc'])
                        })
                        loaded_count += 1
                    else:
                        skipped_count += 1
                        skipped_reasons['ImageNotFound'] = skipped_reasons.get('ImageNotFound', 0) + 1
            
            total_loaded += loaded_count
            total_skipped += skipped_count
            
            # Print detailed skip report
            if skipped_count > 0:
                skip_details = ", ".join([f"{k}:{v}" for k, v in skipped_reasons.items()])
                print(f"  ✓ {run_folder.name}: {loaded_count} samples (skipped {skipped_count}: {skip_details})")
            else:
                print(f"  ✓ {run_folder.name}: {loaded_count} samples")
        
        print("-" * 80)
        print(f"✓ Dataset loaded: {total_loaded} samples total")
        if total_skipped > 0:
            print(f"  Skipped: {total_skipped} frames (non-racing or missing data)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 画像読み込み
        image = Image.open(sample['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        # SOC
        soc = torch.tensor([sample['soc']], dtype=torch.float32)
        
        # ターゲット（drive_torque, steer_angle） ← model.pyの出力順と一致
        target = torch.tensor([sample['drive_torque'], sample['steer_angle']], dtype=torch.float32)
        
        return image, soc, target

# ===========================
# データ変換（Data Augmentation）
# ===========================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===========================
# データセット作成
# ===========================
full_dataset = RobotRaceDataset(data_sources_path, transform=train_transform)

# Train/Val split (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Val dataset transform変更（Augmentationなし）
# Note: random_splitはSubsetを返すので、元のdatasetのtransformは共有される
# そのため、後でvalidation時にval_transformを使うように工夫する

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# ===========================
# DataLoader作成
# ===========================
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# ===========================
# モデル、損失関数、オプティマイザ
# ===========================
model = DrivingNetwork().to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.0001)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
)

print(f"\n✓ Model initialized on {device}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Cell 7: Train model

Training loop with early stopping

In [ ]:
import time
from tqdm import tqdm

# ===========================
# 学習パラメータ
# ===========================
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 0.0001

# ===========================
# Early Stopping クラス
# ===========================
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    
    def __init__(self, patience=15, min_delta=0.0001, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False
        self.best_epoch = 0
    
    def __call__(self, val_loss, epoch):
        if val_loss < self.best_loss - self.min_delta:
            # Improvement
            self.best_loss = val_loss
            self.counter = 0
            self.best_epoch = epoch
            return False
        else:
            # No improvement
            self.counter += 1
            if self.verbose and self.counter > 0:
                print(f"      [EarlyStopping] No improvement for {self.counter}/{self.patience} epochs")
            
            if self.counter >= self.patience:
                self.should_stop = True
                if self.verbose:
                    print(f"      [EarlyStopping] Stopping! Best epoch was {self.best_epoch + 1}")
                return True
        
        return False

# ===========================
# Training Logger（CSV出力）
# ===========================
class TrainingLogger:
    """Log training metrics to CSV file."""
    
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.metrics_history = []
        
        # Write header
        with open(self.log_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                'epoch', 'train_loss', 'train_torque_loss', 'train_steer_loss',
                'val_loss', 'val_torque_loss', 'val_steer_loss',
                'learning_rate', 'is_best', 'timestamp'
            ])
    
    def log(self, epoch, train_metrics, val_metrics, learning_rate, is_best):
        from datetime import datetime
        
        row = {
            'epoch': epoch + 1,
            'train_loss': train_metrics['loss'],
            'train_torque_loss': train_metrics['torque_loss'],
            'train_steer_loss': train_metrics['steer_loss'],
            'val_loss': val_metrics['loss'],
            'val_torque_loss': val_metrics['torque_loss'],
            'val_steer_loss': val_metrics['steer_loss'],
            'learning_rate': learning_rate,
            'is_best': is_best,
            'timestamp': datetime.now().isoformat()
        }
        
        self.metrics_history.append(row)
        
        with open(self.log_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(list(row.values()))

# ===========================
# 学習ループ
# ===========================
train_losses = []
val_losses = []
train_torque_losses = []
train_steer_losses = []
val_torque_losses = []
val_steer_losses = []
learning_rates = []

best_val_loss = float('inf')
best_model_path = iteration_path / "model_best.pth"
final_model_path = iteration_path / "model.pth"

# Early stopping
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=MIN_DELTA)

# Logger
log_path = iteration_path / "training_log.csv"
logger = TrainingLogger(log_path)

print("\n" + "=" * 80)
print("Starting training...")
print("=" * 80)

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # ===========================
    # Training phase
    # ===========================
    model.train()
    train_loss = 0.0
    train_torque_loss = 0.0
    train_steer_loss = 0.0
    
    for images, socs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False):
        images = images.to(device)
        socs = socs.to(device)
        targets = targets.to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images, socs)
        loss = criterion(outputs, targets)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Individual losses
        with torch.no_grad():
            torque_loss = nn.MSELoss()(outputs[:, 0], targets[:, 0])
            steer_loss = nn.MSELoss()(outputs[:, 1], targets[:, 1])
            train_torque_loss += torque_loss.item()
            train_steer_loss += steer_loss.item()
    
    train_loss /= len(train_loader)
    train_torque_loss /= len(train_loader)
    train_steer_loss /= len(train_loader)
    
    # ===========================
    # Validation phase
    # ===========================
    model.eval()
    val_loss = 0.0
    val_torque_loss = 0.0
    val_steer_loss = 0.0
    
    with torch.no_grad():
        for images, socs, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]  ", leave=False):
            images = images.to(device)
            socs = socs.to(device)
            targets = targets.to(device)
            
            outputs = model(images, socs)
            loss = criterion(outputs, targets)
            
            val_loss += loss.item()
            
            torque_loss = nn.MSELoss()(outputs[:, 0], targets[:, 0])
            steer_loss = nn.MSELoss()(outputs[:, 1], targets[:, 1])
            val_torque_loss += torque_loss.item()
            val_steer_loss += steer_loss.item()
    
    val_loss /= len(val_loader)
    val_torque_loss /= len(val_loader)
    val_steer_loss /= len(val_loader)
    
    epoch_time = time.time() - epoch_start
    
    # ===========================
    # Update learning rate
    # ===========================
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    
    # ===========================
    # Check for best model
    # ===========================
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        torch.save(model.state_dict(), final_model_path)
        best_marker = " ✓ NEW BEST"
    else:
        best_marker = ""
    
    # ===========================
    # Log metrics
    # ===========================
    train_metrics = {
        'loss': train_loss,
        'torque_loss': train_torque_loss,
        'steer_loss': train_steer_loss
    }
    val_metrics = {
        'loss': val_loss,
        'torque_loss': val_torque_loss,
        'steer_loss': val_steer_loss
    }
    logger.log(epoch, train_metrics, val_metrics, current_lr, is_best)
    
    # Store for plotting
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_torque_losses.append(train_torque_loss)
    train_steer_losses.append(train_steer_loss)
    val_torque_losses.append(val_torque_loss)
    val_steer_losses.append(val_steer_loss)
    learning_rates.append(current_lr)
    
    # ===========================
    # Print progress
    # ===========================
    print(
        f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.6f} (T:{train_torque_loss:.4f} S:{train_steer_loss:.4f}) | "
        f"Val: {val_loss:.6f} (T:{val_torque_loss:.4f} S:{val_steer_loss:.4f}) | "
        f"LR: {current_lr:.2e} | {epoch_time:.1f}s{best_marker}"
    )
    
    # ===========================
    # Early stopping check
    # ===========================
    if early_stopping(val_loss, epoch):
        print(f"\n⏹️  Early stopping triggered at epoch {epoch+1}")
        break

print("=" * 80)

# ===========================
# 最終モデル保存確認
# ===========================
import shutil
if best_model_path.exists() and not final_model_path.exists():
    shutil.copy(best_model_path, final_model_path)

print("\n✓ Training complete!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Model saved to: {final_model_path}")
print(f"Training log: {log_path}")
print(f"\n📥 次のステップ: このmodel.pthをダウンロードして、ローカルのRobot1/models/にコピーしてください")

---
## Cell 8: Visualize results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ===========================
# 学習曲線プロット
# ===========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = range(1, len(train_losses) + 1)

# Total Loss
axes[0, 0].plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs, val_losses, 'r-', label='Val Loss', linewidth=2)
axes[0, 0].axhline(y=best_val_loss, color='g', linestyle='--', label=f'Best Val Loss ({best_val_loss:.6f})', linewidth=1)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('MSE Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Torque Loss
axes[0, 1].plot(epochs, train_torque_losses, 'b-', label='Train Torque Loss', linewidth=2)
axes[0, 1].plot(epochs, val_torque_losses, 'r-', label='Val Torque Loss', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Torque Loss', fontsize=12)
axes[0, 1].set_title('Drive Torque Loss', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Steer Loss
axes[1, 0].plot(epochs, train_steer_losses, 'b-', label='Train Steer Loss', linewidth=2)
axes[1, 0].plot(epochs, val_steer_losses, 'r-', label='Val Steer Loss', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Steer Loss', fontsize=12)
axes[1, 0].set_title('Steer Angle Loss', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs, learning_rates, 'g-', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = plots_path / "training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print(f"\n✓ Plot saved: {plot_path}")

# ===========================
# サマリー表示
# ===========================
print("\n" + "=" * 80)
print("📊 Training Summary")
print("=" * 80)
print(f"Total epochs: {len(train_losses)}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")

# Overfitting check
overfitting_gap = val_losses[-1] - train_losses[-1]
print(f"\nOverfitting check:")
if overfitting_gap > 0.01:
    print(f"  ⚠️ Possible overfitting detected (gap={overfitting_gap:.6f})")
else:
    print(f"  ✓ No significant overfitting (gap={overfitting_gap:.6f})")

# Learning rate info
print(f"\nLearning rate:")
print(f"  Initial: {learning_rates[0]:.2e}")
print(f"  Final: {learning_rates[-1]:.2e}")

print("\n" + "=" * 80)
print("🎉 All done! 次のステップ:")
print("=" * 80)
print(f"1. {final_model_path} をダウンロード")
print(f"2. ローカルの Robot1/models/ にコピー")
print(f"3. Unity で AI モードをテスト")
print("=" * 80)

# ===========================
# Training logのサンプル表示
# ===========================
if log_path.exists():
    print(f"\n📋 Training log preview (最初の5行と最後の5行):")
    print("-" * 80)
    df = pd.read_csv(log_path)
    print("\n最初の5行:")
    print(df.head().to_string(index=False))
    print("\n最後の5行:")
    print(df.tail().to_string(index=False))
    print("-" * 80)